# Shadow AI Audit Tool — Data Extraction

This notebook pulls metadata for the top 500 models from Hugging Face.

**Instructions:**
1. Run Cell 1 to install the library
2. In Cell 2, replace `hf_your_token_here` with your Hugging Face token
3. Run Cell 2 (takes ~10-15 minutes)
4. Run Cell 3 to download the output JSON file

In [ ]:
!pip install huggingface_hub -q

In [ ]:
import json
import os
import time
from huggingface_hub import HfApi, ModelCard
from huggingface_hub.utils import EntryNotFoundError, RepositoryNotFoundError

TOKEN = "hf_your_token_here"  # <-- PASTE YOUR TOKEN HERE
LIMIT = 500
OUTPUT = "hf_models.json"

api = HfApi(token=TOKEN)
author_cache = {}

known_orgs = set([
    "meta-llama", "openai", "google", "microsoft", "mistralai",
    "deepseek-ai", "stabilityai", "bigscience", "eleutherai",
    "alibaba-nlp", "tiiuae", "01-ai", "Qwen", "cohere",
    "sentence-transformers", "facebook", "huggingface",
    "nvidia", "salesforce", "databricks", "allenai",
    "mosaicml", "together", "lmsys", "THUDM", "baichuan-inc",
    "internlm", "HuggingFaceH4", "Open-Orca", "teknium",
    "NousResearch", "berkeley-nest", "amazon", "apple",
])

known_verified = set([
    "meta-llama", "openai", "google", "microsoft", "mistralai",
    "deepseek-ai", "stabilityai", "alibaba-nlp", "tiiuae",
    "Qwen", "nvidia", "salesforce", "facebook", "huggingface",
    "apple", "amazon", "databricks", "allenai", "sentence-transformers",
])


def get_card_text(model_id):
    try:
        card = ModelCard.load(model_id, token=TOKEN)
        return card.text or ""
    except:
        return ""


def get_author_stats(author):
    if author in author_cache:
        return author_cache[author]
    try:
        models = list(api.list_models(author=author, limit=200))
        total = len(models)
        licensed = 0
        for m in models:
            tags = getattr(m, "tags", None) or []
            for t in tags:
                if t.startswith("license:"):
                    licensed += 1
                    break
        pct = round(licensed / total, 2) if total > 0 else 0
        result = {"model_count": total, "license_pct": pct}
    except:
        result = {"model_count": 0, "license_pct": 0}
    author_cache[author] = result
    return result


def is_org(author):
    if author in known_orgs:
        return True
    has_hyphen = "-" in str(author)
    has_digit = any(c.isdigit() for c in str(author))
    if author and has_hyphen and not has_digit:
        return True
    return False


print(f"Fetching top {LIMIT} models by downloads...")
models_list = list(api.list_models(
    sort="downloads",
    direction=-1,
    limit=LIMIT,
    full=True,
    cardData=True,
))
print(f"Retrieved {len(models_list)} models.\n")

results = []
start = time.time()

for i, m in enumerate(models_list):
    model_id = m.modelId or m.id
    author = getattr(m, "author", None)
    if not author and "/" in model_id:
        author = model_id.split("/")[0]
    if not author:
        author = model_id

    if (i + 1) % 25 == 0 or i == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        remaining = (len(models_list) - i - 1) / rate if rate > 0 else 0
        print(f"[{i+1}/{len(models_list)}] {model_id} ({rate:.1f}/sec, ~{remaining:.0f}s left)")

    card_text = get_card_text(model_id)
    stats = get_author_stats(author)

    license_id = None
    license_name = None
    license_link = None
    cd = getattr(m, "card_data", None)
    if cd:
        license_id = getattr(cd, "license", None)
        license_name = getattr(cd, "license_name", None)
        license_link = getattr(cd, "license_link", None)
    if not license_id:
        tags = getattr(m, "tags", []) or []
        for t in tags:
            if t.startswith("license:"):
                license_id = t.replace("license:", "")
                break

    lm = getattr(m, "lastModified", None)
    if lm is None:
        lm = getattr(m, "last_modified", None)
    if lm and hasattr(lm, "isoformat"):
        last_modified = lm.isoformat()
    elif lm:
        last_modified = str(lm)
    else:
        last_modified = None

    gated = getattr(m, "gated", False)
    if gated in ("auto", "manual"):
        gated = True
    else:
        gated = bool(gated)

    base_model = None
    if cd:
        bm = getattr(cd, "base_model", None)
        if isinstance(bm, list):
            base_model = bm[0] if bm else None
        else:
            base_model = bm

    tags = list(getattr(m, "tags", []) or [])

    record = {
        "id": model_id,
        "author": author,
        "license": license_id,
        "license_name": license_name,
        "license_link": license_link,
        "pipeline_tag": getattr(m, "pipeline_tag", None),
        "downloads": getattr(m, "downloads", 0) or 0,
        "likes": getattr(m, "likes", 0) or 0,
        "last_modified": last_modified,
        "tags": tags,
        "gated": gated,
        "base_model": base_model,
        "is_org": is_org(author),
        "verified": author in known_verified,
        "org_model_count": stats["model_count"],
        "org_license_pct": stats["license_pct"],
        "card_text": card_text,
    }
    results.append(record)

    if (i + 1) % 10 == 0:
        time.sleep(0.3)

elapsed_total = time.time() - start
print(f"\nDone. {len(results)} models in {elapsed_total:.0f}s.")

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)

size = os.path.getsize(OUTPUT) / (1024 * 1024)
print(f"Saved to {OUTPUT} ({size:.1f} MB)")

print(f"\nWith model cards: {sum(1 for r in results if r['card_text'])}")
print(f"No license: {sum(1 for r in results if not r['license'])}")
print(f"Gated: {sum(1 for r in results if r['gated'])}")

lics = {}
for r in results:
    k = r["license"] or "none"
    lics[k] = lics.get(k, 0) + 1
print(f"\nTop licenses:")
for k, v in sorted(lics.items(), key=lambda x: -x[1])[:10]:
    print(f"  {k:30s} {v:>5d}")

In [ ]:
from google.colab import files
files.download('hf_models.json')